# Outliers and impossible values: detect, then decide

_Data Wrangling_

**Find the points that sit far from the rest, then ask: is it a data error, or a real extreme worth keeping?**

An outlier is a value that sits far from the bulk of the data. Detecting one is easy; the
       hard part is the decision that follows. The same far-out point can be a typo to delete or the
       single most valuable row in the table &mdash; and you cannot tell which from the number alone. You
       have to ask what it means.

       So split the work in two. First detect &mdash; use a rule to flag points that are unusually
       far out. Then decide &mdash; look at each flagged point and classify it:

---

This notebook is a practice scaffold for the **Outliers and impossible values: detect, then decide** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas + scikit-learn

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.datasets import load_breast_cancer

# Real bundled data; treat 'mean area' as a right-skewed, count-like column.
df = load_breast_cancer(as_frame=True).frame
col = 'mean area'
x = df[col]

# === 1. z-score rule: flag |z| > 3 ===
z = (x - x.mean()) / x.std(ddof=0)
df['z_outlier'] = z.abs() > 3

# === 2. IQR (Inter-Quartile Range) fence ===
q1, q3 = x.quantile(0.25), x.quantile(0.75)
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr     # the fence
df['iqr_outlier'] = (x < lo) | (x > hi)
print(f'IQR fence: [{lo:.0f}, {hi:.0f}] -> {df.iqr_outlier.sum()} flagged')

# === 3. Robust z-score (median / MAD) -- good for skewed columns ===
med = x.median()
mad = (x - med).abs().median()              # Median Absolute Deviation
robust_z = 0.6745 * (x - med) / mad         # 0.6745 makes it ~comparable to z
df['robust_outlier'] = robust_z.abs() > 3.5

# === 4. Multivariate: Mahalanobis distance over a few correlated columns ===
feats = ['mean area', 'mean perimeter', 'mean radius']
M = df[feats].values
center = M.mean(axis=0)
cov_inv = np.linalg.inv(np.cov(M, rowvar=False))
d2 = np.array([stats.mahalanobis(r, center, cov_inv) ** 2 for r in M])
# chi-square cutoff (df = #features) at the 0.999 level
df['maha_outlier'] = d2 > stats.chi2.ppf(0.999, df=len(feats))

# === 4b. Multivariate: IsolationForest (and LOF is a drop-in alternative) ===
iso = IsolationForest(contamination=0.02, random_state=0)
df['iso_outlier'] = iso.fit_predict(df[feats].values) == -1   # -1 == outlier
# from sklearn.neighbors import LocalOutlierFactor
# df['lof_outlier'] = LocalOutlierFactor(n_neighbors=20).fit_predict(df[feats]) == -1

# === DECIDE: sanity-check flags against a domain range ===
# A 'mean area' cannot be negative or absurdly huge -> those would be ERRORS.
IMPOSSIBLE = (x <= 0) | (x > 5000)          # domain rule, not a statistic
errors   = df[IMPOSSIBLE]                    # fix or DROP these
extremes = df[df.iqr_outlier & ~IMPOSSIBLE] # real extremes -> usually KEEP
print(f'{len(errors)} impossible (error) rows, {len(extremes)} real extremes')

# === HANDLE option: winsorize / cap real extremes to the IQR fence ===
df[col + '_capped'] = x.clip(lower=lo, upper=hi)
# trees: leave raw. linear / distance models: use the capped (or log) column.
df[col + '_log'] = np.log1p(x)              # transform alternative to capping

## Visualize the data & results

_On a real right-skewed feature ('mean area' from load_breast_cancer), which points does the IQR (Inter-Quartile Range) fence flag as outliers?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer

d = load_breast_cancer()
fi = list(d.feature_names).index('mean area')
area = d.data[:, fi]                                  # 569 real measurements

rng = np.random.RandomState(0)
area = np.sort(area[rng.choice(len(area), 60, replace=False)])   # subsample 60

# IQR (Inter-Quartile Range) fence
q1, q3 = np.percentile(area, [25, 75])
iqr = q3 - q1
hi = q3 + 1.5 * iqr
print(f'Q1={q1:.0f} Q3={q3:.0f} IQR={iqr:.0f} upper fence={hi:.0f}')
# -> Q1=429 Q3=752 IQR=323 upper fence=1237
out = area > hi
print('IQR outliers:', out.sum(), '->', np.round(area[out]).astype(int))
# -> 6 -> [1319 1326 1364 1386 1479 1491]

# z-score flags NONE of them -- the skewed tail inflates the std and masks them
z = (area - area.mean()) / area.std()
print('max |z| =', round(np.abs(z).max(), 2), '-> |z|>3 count:', (np.abs(z) > 3).sum())
# -> max |z| = 2.51 -> |z|>3 count: 0

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A column of customer ages contains a value of 200 and another of 104. Both are flagged by an IQR fence. Should they get the same treatment? Walk through the decision.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Detect: both sit past the upper IQR fence, so both are candidates. — _The fence only flags "far out" &mdash; it cannot tell error from extreme on its own._
- Apply a domain range check: a human age cannot exceed about 122 (the record). — _200 is impossible &mdash; a data error (a typo or sentinel). 104 is rare but physically real._
- Fix or remove the 200 (treat as missing, or correct it); keep the 104. — _Deleting the 104 would throw away a genuine elderly customer &mdash; possibly an important segment._

**Answer:** No &mdash; same flag, different action. The 200 is an impossible value (above the human limit), so it is a data error: fix it or remove the row. The 104 is a real extreme: keep it. The IQR fence detects both; only the domain range check separates error from genuine extreme.

</details>

**Problem 2.** On a right-skewed income column, the plain z-score rule ($|z|\gt 3$) flags almost nothing, even though there are obvious extreme earners. Why does it fail, and what detectors should you use instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that the mean and standard deviation are sums over all values. — _A few huge incomes pull the mean up and inflate $s$, so every z-score shrinks._
- Realize the outliers mask themselves: their own size enlarges $s$, dropping their $|z|$ below 3. — _The rule assumes a symmetric bell shape; income is heavy-tailed, so the assumption breaks._
- Switch to the IQR fence or the robust z (median / MAD), which use ranks, not the mean. — _Quartiles and the median are unmoved by a few extremes, so the fence stays put and flags them._

**Answer:** The z-score assumes a symmetric column. On skewed income, the extreme earners inflate the mean and standard deviation, shrinking every z-score &mdash; the outliers mask themselves. Use the IQR (Inter-Quartile Range) fence or the robust z-score (median / MAD), both built on ranks, or first log-transform the column so it becomes roughly symmetric.

</details>

**Problem 3.** You have a genuine, verified extreme value (a real record-breaking transaction). You will train two models on this data: a gradient-boosted tree and a linear regression. How should you handle the point for each?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Confirm it is a real extreme, not an error, via a domain check. — _Verified-real means "do not delete" &mdash; it carries signal._
- For the tree model, leave it. Trees split on thresholds, so how far out the value is barely matters. — _A split at "amount > t" treats 10x and 100x the same; the magnitude does not distort the fit._
- For the linear model, cap/winsorize or log-transform so the point does not dominate the slope. — _Linear / distance models square the residual, so one far point can swing the whole fit._

**Answer:** Because the point is real, never delete it. For the tree, leave it &mdash; trees split on thresholds and are nearly immune to how extreme a value is. For the linear regression, cap / winsorize it or log-transform the column, since linear and distance-based models square residuals and let one far point dominate. The handling choice depends on the model.

</details>